# Logistic regression patient records

Original experiment from [Machine_Learning](https://github.com/Muhammad-Huzifa/Machine_Learning/blob/41d456774a4e2c52e7b6d701ea244eec7b903a88/Logistic_Regression_From_Scratch_With_Real_Data_Set.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd


df = pd.read_csv(str(TABULAR / 'patient_records.csv'))

df = df.drop(columns=['Name'])

X = df.drop('Dead', axis=1).values
y = df['Dead'].values.reshape(-1, 1)

df

## Normalization

In [ ]:
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)
X = np.hstack([np.ones((X.shape[0], 1)), X])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Sigmoid , Cost , and Gradient Descent

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def compute_cost(X, y, weights):
    m = len(y)
    z = np.dot(X, weights)
    h = sigmoid(z)
    cost = (-1/m) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h))
    return cost

def gradient_descent(X, y, weights, learning_rate, iterations):
    m = len(y)
    cost_history = np.zeros((iterations, 1))

    for i in range(iterations):
        z = np.dot(X, weights)
        h = sigmoid(z)
        gradient = np.dot(X.T, (h - y)) / m
        weights -= learning_rate * gradient


        cost_history[i] = compute_cost(X, y, weights)

    return weights, cost_history

## Initialization and Prediction

In [ ]:

learning_rate = 0.01
iterations = 10000
weights = np.zeros((X_train.shape[1], 1))

weights, cost_history = gradient_descent(X_train, y_train, weights, learning_rate, iterations)

def predict(X, weights):
    z = np.dot(X, weights)
    h = sigmoid(z)
    return [1 if i > 0.5 else 0 for i in h]

y_pred = predict(X_test, weights)

y_pred = np.array(y_pred).reshape(-1, 1)





### Accuracy , Precision , and Recall 

In [ ]:
accuracy = np.mean(y_pred == y_test) * 100
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))
print(TP,TN,FP,FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# Output the results
print(f"Accuracy: {accuracy:.2f}%")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1_score:.2f}")
